# 🌊 Day 02: Graph BFS — Shortest Path & Multi-Source

---

## Why BFS for Shortest Path?

BFS explores level by level. In an unweighted graph, the first time you reach a node is the shortest path to it. That's guaranteed by the level-by-level expansion.

**Multi-source BFS:** Start BFS from multiple sources simultaneously. Classic problems: rotting oranges, 01 matrix.

---

In [ ]:
from collections import deque, defaultdict

## 🎯 Problem 1: Number of Islands (LC #200)

Given a grid of `'1'` (land) and `'0'` (water), count the number of islands. An island is a group of connected `'1'`s.

**Approach:** Scan the grid. When you find a `'1'`, start BFS/DFS to mark the entire island as visited. Count how many times you start a new traversal.

In [ ]:
def numIslands(grid):
    """
    BFS from each unvisited '1'. Each BFS = one island.
    Time: O(m*n), Space: O(m*n)
    """
    if not grid:
        return 0
    
    rows, cols = len(grid), len(grid[0])
    islands = 0
    
    def bfs(r, c):
        queue = deque([(r, c)])
        grid[r][c] = '0'  # Mark visited by sinking
        while queue:
            row, col = queue.popleft()
            for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
                nr, nc = row + dr, col + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == '1':
                    grid[nr][nc] = '0'
                    queue.append((nr, nc))
    
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                bfs(r, c)
                islands += 1
    
    return islands

grid = [
    ['1','1','0','0','0'],
    ['1','1','0','0','0'],
    ['0','0','1','0','0'],
    ['0','0','0','1','1']
]
print(f"Number of islands: {numIslands(grid)}")  # 3

**Tip:** Instead of a separate `visited` set, we "sink" the island by changing `'1'` to `'0'`. Saves space and is a common grid trick.

---

## 🎯 Problem 2: Rotting Oranges (LC #994)

Multi-source BFS. All rotten oranges (value 2) spread rot simultaneously each minute. How many minutes until all oranges are rotten?

**Key insight:** Put ALL rotten oranges into the queue at the start. Each BFS level = one minute.

In [ ]:
def orangesRotting(grid):
    """
    Multi-source BFS: all rotten oranges spread simultaneously.
    Time: O(m*n), Space: O(m*n)
    """
    rows, cols = len(grid), len(grid[0])
    queue = deque()
    fresh = 0
    
    # Collect all initial rotten oranges and count fresh
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                queue.append((r, c))
            elif grid[r][c] == 1:
                fresh += 1
    
    if fresh == 0:
        return 0
    
    minutes = 0
    while queue:
        # Process all oranges that rot at current minute
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    queue.append((nr, nc))
        minutes += 1
    
    # -1 because we count one extra after the last spread
    return minutes - 1 if fresh == 0 else -1

grid = [[2,1,1],[1,1,0],[0,1,1]]
print(f"Minutes: {orangesRotting(grid)}")  # 4

---

## 🎯 Problem 3: Word Ladder (LC #127)

Transform one word to another by changing one letter at a time, each intermediate word must be in the dictionary. Find shortest transformation length.

```
"hit" → "hot" → "dot" → "dog" → "cog" = length 5
```

**This is BFS on an implicit graph** where each word is a node, and edges connect words that differ by one letter.

In [ ]:
def ladderLength(beginWord, endWord, wordList):
    """
    BFS, each level = one transformation.
    Build pattern map: h*t → [hit, hot, hat]
    Time: O(m² * n), Space: O(m² * n)
    """
    wordSet = set(wordList)
    if endWord not in wordSet:
        return 0
    
    # Build pattern → words mapping for quick neighbor lookup
    patterns = defaultdict(list)
    for word in wordSet:
        for i in range(len(word)):
            pattern = word[:i] + '*' + word[i+1:]
            patterns[pattern].append(word)
    
    queue = deque([(beginWord, 1)])
    visited = {beginWord}
    
    while queue:
        word, length = queue.popleft()
        
        for i in range(len(word)):
            pattern = word[:i] + '*' + word[i+1:]
            for neighbor in patterns[pattern]:
                if neighbor == endWord:
                    return length + 1
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, length + 1))
    
    return 0

print(ladderLength("hit", "cog", ["hot","dot","dog","lot","log","cog"]))  # 5

---

## 🗺️ Graph BFS Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  GRAPH BFS PATTERNS                                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. GRID TRAVERSAL: Each '1' triggers BFS, sink to mark visited    ║
║  2. MULTI-SOURCE: Put ALL sources in queue initially               ║
║     Each BFS level = one time step (rotting oranges, 01 matrix)    ║
║  3. IMPLICIT GRAPH: Nodes = states, edges = valid transitions      ║
║     Word ladder: words are nodes, 1-letter diff = edge             ║
║                                                                      ║
║  BFS GUARANTEES: First visit = shortest path (unweighted)           ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 200 | Number of Islands | Medium | Grid BFS/DFS, sink to mark |
| 994 | Rotting Oranges | Medium | Multi-source BFS |
| 127 | Word Ladder | Hard | Implicit graph BFS |
| 542 | 01 Matrix | Medium | Multi-source BFS from 0s |
| 1091 | Shortest Path in Binary Matrix | Medium | BFS 8-directional |
| 286 | Walls and Gates | Medium | Multi-source BFS from gates |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 BFS = shortest path in unweighted graphs. Always.             │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Multi-source: all sources in queue at start = simultaneous BFS│
├──────────────────────────────────────────────────────────────────────┤
│  📌 Grid trick: sink visited cells (change value) instead of set  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Implicit graph: states = nodes, transitions = edges           │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Graph DFS

Connected components, cycle detection, Pacific Atlantic water flow.